In [ ]:
import os
import torch
import sys

# Check GPU & PyTorch (Sanity Check)
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
!nvidia-smi

# Configuration
REPO_URL = "https://github.com/fabianandresgrob/gaussian-splatting.git"
REPO_BRANCH = "view-selection"
WORKING_DIR = "/content/gsplat_workspace"
REPO_NAME = REPO_URL.split("/")[-1].replace(".git", "")

# 4. Create Workspace & Clone
os.makedirs(WORKING_DIR, exist_ok=True)
os.chdir(WORKING_DIR)

if not os.path.exists(REPO_NAME):
    print(f"--- Cloning {REPO_NAME} ---")
    !git clone {REPO_URL} --recursive
    os.chdir(REPO_NAME)
    !git checkout {REPO_BRANCH}
else:
    print(f"--- Repository exists. Pulling latest (forced) ---")
    os.chdir(REPO_NAME)
    # We don't care about local changes, just overwrite
    !git reset --hard
    !git pull --force
    !git checkout {REPO_BRANCH}
    !git submodule update --init --recursive

REPO_PATH = os.getcwd()
print(f"SUCCESS: Repo ready at {REPO_PATH}")

In [ ]:
# Cell 2: Add to path
module_path = '/kaggle/working/gaussian-splatting/'
if module_path not in sys.path:
    sys.path.append(module_path)
    print(f"Added {module_path} to sys.path")
else:
    print(f"{module_path} already in sys.path")

In [ ]:
# Cell 3: Install Dependencies
# Kaggle has: torch, torchvision, numpy, pillow, matplotlib pre-installed
# We need to add the missing ones:

!pip install -q tqdm plyfile scipy scikit-learn opencv-python joblib tensorboard

import setup_utils
import pandas as pd
import numpy as np

# Compile CUDA extensions (patches + builds diff-gaussian-rasterization & simple-knn)
setup_utils.setup_environment(os.getcwd())

In [ ]:
# Cell 4: Launch TensorBoard
OUTPUT_ROOT = "/kaggle/working/3DGS_Results"
os.makedirs(OUTPUT_ROOT, exist_ok=True)

%load_ext tensorboard
%tensorboard --logdir {OUTPUT_ROOT}

In [ ]:
# Cell 5: Define Experiments & Run on DUAL GPUs in Parallel
import subprocess
import concurrent.futures
import time
import threading
from queue import Queue

# ============== CONFIGURATION ==============
DATA_ROOT = "/kaggle/input/scannetpp/data"  # Adjust to your Kaggle dataset path

SCENES = ["0c5385e84b"]  # Add more scenes as needed
STRATEGIES = ["random", "fixed_prob"]  # Add: "epoch_based", "clustering", "no_replace"
SEEDS = [0, 1, 2, 3]

# Training configuration
TRAINING_CONFIG = {
    "iterations": 10000,
    "test_iterations": [1000, 2000, 4000, 6000, 8000, 10000],
    "save_iterations": [10000],
    "resolution": 2,  # Half resolution for faster training
}

# Strategy-specific configurations
STRATEGY_CONFIGS = {
    "random": "{}",
    "fixed_prob": '{"temperature": 1.0, "distance_weight": 0.5, "diversity_weight": 0.5}',
    "epoch_based": '{"penalty_strength": 1.0}',
    "clustering": '{"n_clusters": 10, "temperature": 1.0}',
    "no_replace": "{}",
}

# ============== BUILD EXPERIMENT LIST ==============
experiments = []
for scene in SCENES:
    scene_path = os.path.join(DATA_ROOT, scene, "dslr")
    for strategy in STRATEGIES:
        exp_name = f"{scene}_{strategy}"
        view_config = STRATEGY_CONFIGS.get(strategy, "{}")
        for seed in SEEDS:
            experiments.append({
                "scene": scene,
                "scene_path": scene_path,
                "strategy": strategy,
                "seed": seed,
                "exp_name": exp_name,
                "view_config": view_config,
            })

print(f"Total experiments: {len(experiments)}")
print(f"Will run on 2 GPUs in parallel (2x speedup)")

# ============== PARALLEL EXECUTION FUNCTION ==============
def run_experiment_subprocess(exp, gpu_id):
    """Run a single experiment on a specific GPU using subprocess."""
    output_path = os.path.join(OUTPUT_ROOT, exp['exp_name'], f"seed_{exp['seed']}")
    
    # Skip if already completed
    if os.path.exists(os.path.join(output_path, "final_results.json")):
        print(f"[GPU {gpu_id}] SKIP: {exp['exp_name']} seed {exp['seed']} (already done)")
        return {"status": "skipped", "exp": exp}
    
    print(f"[GPU {gpu_id}] START: {exp['exp_name']} seed {exp['seed']}")
    
    # Build command
    cmd = [
        "python", "train_gsplat.py",
        "-s", exp['scene_path'],
        "-m", output_path,
        "--iterations", str(TRAINING_CONFIG['iterations']),
        "--test_iterations", *[str(i) for i in TRAINING_CONFIG['test_iterations']],
        "--save_iterations", *[str(i) for i in TRAINING_CONFIG['save_iterations']],
        "--view_selection_strategy", exp['strategy'],
        "--view_selection_config", exp['view_config'],
        "--view_selection_seed", str(exp['seed']),
        "-r", str(TRAINING_CONFIG['resolution']),
        "--data_device", "cuda",
    ]
    
    # Set environment to use specific GPU
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
    
    try:
        start_time = time.time()
        result = subprocess.run(
            cmd,
            cwd=REPO_PATH,
            env=env,
            capture_output=True,
            text=True,
            timeout=7200  # 2 hour timeout
        )
        elapsed = time.time() - start_time
        
        # Save console log
        os.makedirs(output_path, exist_ok=True)
        with open(os.path.join(output_path, "console_log.txt"), "w") as f:
            f.write(f"=== STDOUT ===\n{result.stdout}\n\n=== STDERR ===\n{result.stderr}")
        
        if result.returncode == 0:
            print(f"[GPU {gpu_id}] SUCCESS: {exp['exp_name']} seed {exp['seed']} ({elapsed:.1f}s)")
            return {"status": "success", "exp": exp, "time": elapsed}
        else:
            print(f"[GPU {gpu_id}] FAILED: {exp['exp_name']} seed {exp['seed']}")
            return {"status": "failed", "exp": exp, "error": result.stderr[-500:]}
            
    except subprocess.TimeoutExpired:
        print(f"[GPU {gpu_id}] TIMEOUT: {exp['exp_name']} seed {exp['seed']}")
        return {"status": "timeout", "exp": exp}
    except Exception as e:
        print(f"[GPU {gpu_id}] ERROR: {exp['exp_name']} seed {exp['seed']}: {e}")
        return {"status": "error", "exp": exp, "error": str(e)}

# ============== GPU WORKER (pulls from shared queue) ==============
def gpu_worker(gpu_id, experiment_queue, results_list, results_lock):
    """Worker that continuously pulls experiments from queue until empty."""
    while True:
        try:
            exp = experiment_queue.get_nowait()
        except:
            # Queue is empty, worker is done
            break
        
        result = run_experiment_subprocess(exp, gpu_id)
        
        with results_lock:
            results_list.append(result)
        
        experiment_queue.task_done()

# ============== RUN EXPERIMENTS ON 2 GPUS ==============
NUM_GPUS = min(2, torch.cuda.device_count())
print(f"\nRunning on {NUM_GPUS} GPU(s)...")

results = []

if NUM_GPUS >= 2:
    # Use a shared queue - whichever GPU finishes first grabs the next experiment
    experiment_queue = Queue()
    for exp in experiments:
        experiment_queue.put(exp)
    
    results_lock = threading.Lock()
    
    print(f"Work queue: {experiment_queue.qsize()} experiments")
    print(f"Both GPUs will stay busy until queue is empty!")
    
    # Start worker threads for each GPU
    threads = []
    for gpu_id in range(NUM_GPUS):
        t = threading.Thread(target=gpu_worker, args=(gpu_id, experiment_queue, results, results_lock))
        t.start()
        threads.append(t)
    
    # Wait for all threads to complete
    for t in threads:
        t.join()

else:
    # Single GPU fallback
    print("Only 1 GPU available, running sequentially...")
    for exp in experiments:
        results.append(run_experiment_subprocess(exp, 0))

# Summary
print("\n" + "="*50)
print("EXECUTION SUMMARY")
print("="*50)
success = sum(1 for r in results if r['status'] == 'success')
skipped = sum(1 for r in results if r['status'] == 'skipped')
failed = sum(1 for r in results if r['status'] in ['failed', 'error', 'timeout'])
print(f"Success: {success}")
print(f"Skipped: {skipped}")
print(f"Failed:  {failed}")

In [ ]:
# Cell 6: Aggregate Results
import experiment_lib

results = []
for scene in SCENES:
    for strategy in STRATEGIES:
        rep = experiment_lib.aggregate_results(OUTPUT_ROOT, scene, strategy)
        if rep: 
            results.append(rep)

df = pd.DataFrame(results)
print("\n" + "="*70)
print("FINAL AGGREGATED REPORT (Mean +/- Std across seeds)")
print("="*70)
print(df.to_string(index=False))
df.to_csv(os.path.join(OUTPUT_ROOT, "final_experiment_report.csv"), index=False)
print(f"\n[Saved] {os.path.join(OUTPUT_ROOT, 'final_experiment_report.csv')}")

In [ ]:
# Cell 7: Detailed Per-Seed Analysis
import json
import glob

def load_all_results(output_root, scenes, strategies):
    """Load all individual seed results into a detailed DataFrame."""
    all_data = []
    
    for scene in scenes:
        for strategy in strategies:
            exp_folder = os.path.join(output_root, f"{scene}_{strategy}")
            seed_dirs = sorted(glob.glob(os.path.join(exp_folder, "seed_*")))
            
            for seed_dir in seed_dirs:
                seed = int(os.path.basename(seed_dir).split("_")[1])
                
                final_file = os.path.join(seed_dir, "final_results.json")
                if os.path.exists(final_file):
                    with open(final_file, 'r') as f:
                        final = json.load(f)
                    
                    history_file = os.path.join(seed_dir, "metrics_history.json")
                    peak_psnr = final['mean_psnr']
                    peak_iter = None
                    
                    if os.path.exists(history_file):
                        with open(history_file, 'r') as f:
                            history = json.load(f)
                        if history:
                            test_psnrs = [(h['iteration'], h['test']['PSNR']) for h in history if 'test' in h]
                            if test_psnrs:
                                peak_iter, peak_psnr = max(test_psnrs, key=lambda x: x[1])
                    
                    all_data.append({
                        'Scene': scene,
                        'Strategy': strategy,
                        'Seed': seed,
                        'Final_PSNR': final['mean_psnr'],
                        'Final_SSIM': final['mean_ssim'],
                        'Final_LPIPS': final['mean_lpips'],
                        'Peak_PSNR': peak_psnr,
                        'Peak_Iter': peak_iter,
                    })
    
    return pd.DataFrame(all_data)

df_detailed = load_all_results(OUTPUT_ROOT, SCENES, STRATEGIES)
print("Per-Seed Results:")
print(df_detailed.to_string(index=False))
print(f"\nTotal runs: {len(df_detailed)}")

In [ ]:
# Cell 8: Statistical Comparison
def compare_strategies(df_detailed):
    """Compare strategies with proper statistics."""
    print("\n" + "="*70)
    print("STRATEGY COMPARISON (per scene)")
    print("="*70)
    
    for scene in df_detailed['Scene'].unique():
        print(f"\n[Scene] {scene}")
        print("-" * 50)
        
        scene_data = df_detailed[df_detailed['Scene'] == scene]
        
        summary = scene_data.groupby('Strategy').agg({
            'Final_PSNR': ['mean', 'std'],
            'Peak_PSNR': ['mean', 'std'],
            'Final_SSIM': ['mean', 'std'],
            'Final_LPIPS': ['mean', 'std'],
            'Peak_Iter': 'mean',
        }).round(4)
        
        summary.columns = ['_'.join(col).strip() for col in summary.columns.values]
        
        for strategy in summary.index:
            row = summary.loc[strategy]
            print(f"\n  Strategy: {strategy}")
            print(f"    Final PSNR:  {row['Final_PSNR_mean']:.3f} +/- {row['Final_PSNR_std']:.3f}")
            print(f"    Peak PSNR:   {row['Peak_PSNR_mean']:.3f} +/- {row['Peak_PSNR_std']:.3f} (at iter {row['Peak_Iter_mean']:.0f})")
            print(f"    Final SSIM:  {row['Final_SSIM_mean']:.4f} +/- {row['Final_SSIM_std']:.4f}")
            print(f"    Final LPIPS: {row['Final_LPIPS_mean']:.4f} +/- {row['Final_LPIPS_std']:.4f}")
        
        best_strategy = summary['Peak_PSNR_mean'].idxmax()
        print(f"\n  >> Best strategy (by Peak PSNR): {best_strategy}")

compare_strategies(df_detailed)

In [ ]:
# Cell 9: Plot Training Curves
import matplotlib.pyplot as plt

def plot_training_curves(output_root, scenes, strategies, metric='PSNR'):
    """Plot training curves for all strategies."""
    fig, axes = plt.subplots(1, len(scenes), figsize=(7*len(scenes), 5))
    if len(scenes) == 1:
        axes = [axes]
    
    colors = {'random': 'blue', 'fixed_prob': 'orange', 'epoch_based': 'green', 
              'clustering': 'red', 'no_replace': 'purple'}
    
    for ax, scene in zip(axes, scenes):
        ax.set_title(f'Scene: {scene}')
        ax.set_xlabel('Iteration')
        ax.set_ylabel(f'Test {metric}')
        
        for strategy in strategies:
            exp_folder = os.path.join(output_root, f"{scene}_{strategy}")
            seed_dirs = sorted(glob.glob(os.path.join(exp_folder, "seed_*")))
            
            all_curves = []
            for seed_dir in seed_dirs:
                history_file = os.path.join(seed_dir, "metrics_history.json")
                if os.path.exists(history_file):
                    with open(history_file, 'r') as f:
                        history = json.load(f)
                    if history:
                        iters = [h['iteration'] for h in history if 'test' in h]
                        values = [h['test'][metric] for h in history if 'test' in h]
                        all_curves.append((iters, values))
            
            if all_curves:
                for iters, values in all_curves:
                    ax.plot(iters, values, color=colors.get(strategy, 'gray'), 
                           alpha=0.2, linewidth=1)
                
                all_iters = all_curves[0][0]
                mean_values = np.mean([[v for v in curve[1]] for curve in all_curves], axis=0)
                ax.plot(all_iters, mean_values, color=colors.get(strategy, 'gray'), 
                       linewidth=2, label=strategy)
        
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_root, 'training_curves.png'), dpi=150)
    plt.show()
    print(f"[Saved] {os.path.join(output_root, 'training_curves.png')}")

plot_training_curves(OUTPUT_ROOT, SCENES, STRATEGIES)

In [ ]:
# Cell 10: Bar Chart Comparison
def plot_bar_comparison(df_detailed, metric='Peak_PSNR'):
    """Create bar chart comparing strategies across scenes."""
    fig, ax = plt.subplots(figsize=(10, 6))
    
    summary = df_detailed.groupby(['Scene', 'Strategy'])[metric].agg(['mean', 'std']).reset_index()
    
    scenes = summary['Scene'].unique()
    strategies = summary['Strategy'].unique()
    x = np.arange(len(scenes))
    width = 0.8 / len(strategies)
    
    colors = {'random': '#1f77b4', 'fixed_prob': '#ff7f0e', 'epoch_based': '#2ca02c', 
              'clustering': '#d62728', 'no_replace': '#9467bd'}
    
    for i, strategy in enumerate(strategies):
        data = summary[summary['Strategy'] == strategy]
        means = [data[data['Scene'] == s]['mean'].values[0] if s in data['Scene'].values else 0 for s in scenes]
        stds = [data[data['Scene'] == s]['std'].values[0] if s in data['Scene'].values else 0 for s in scenes]
        
        ax.bar(x + i*width, means, width, label=strategy, 
               color=colors.get(strategy, 'gray'), yerr=stds, capsize=3)
    
    ax.set_xlabel('Scene')
    ax.set_ylabel(metric.replace('_', ' '))
    ax.set_title(f'{metric.replace("_", " ")} by Strategy (Mean +/- Std)')
    ax.set_xticks(x + width * (len(strategies)-1) / 2)
    ax.set_xticklabels(scenes, rotation=45, ha='right')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_ROOT, f'{metric}_comparison.png'), dpi=150)
    plt.show()

plot_bar_comparison(df_detailed, 'Peak_PSNR')
plot_bar_comparison(df_detailed, 'Final_LPIPS')

In [ ]:
# Cell 11: Export LaTeX Table
def export_latex_table(df_detailed, output_path):
    """Export results as LaTeX table for paper."""
    
    summary = df_detailed.groupby(['Scene', 'Strategy']).agg({
        'Peak_PSNR': ['mean', 'std'],
        'Final_SSIM': ['mean', 'std'],
        'Final_LPIPS': ['mean', 'std'],
    }).round(3)
    
    latex_rows = []
    latex_rows.append(r"\begin{table}[h]")
    latex_rows.append(r"\centering")
    latex_rows.append(r"\caption{View Selection Strategy Comparison}")
    latex_rows.append(r"\begin{tabular}{llccc}")
    latex_rows.append(r"\toprule")
    latex_rows.append(r"Scene & Strategy & PSNR $\uparrow$ & SSIM $\uparrow$ & LPIPS $\downarrow$ \\")
    latex_rows.append(r"\midrule")
    
    for (scene, strategy), row in summary.iterrows():
        psnr_str = f"{row[('Peak_PSNR', 'mean')]:.2f} +/- {row[('Peak_PSNR', 'std')]:.2f}"
        ssim_str = f"{row[('Final_SSIM', 'mean')]:.3f} +/- {row[('Final_SSIM', 'std')]:.3f}"
        lpips_str = f"{row[('Final_LPIPS', 'mean')]:.3f} +/- {row[('Final_LPIPS', 'std')]:.3f}"
        latex_rows.append(f"{scene} & {strategy} & {psnr_str} & {ssim_str} & {lpips_str} \\\\")
    
    latex_rows.append(r"\bottomrule")
    latex_rows.append(r"\end{tabular}")
    latex_rows.append(r"\label{tab:view_selection}")
    latex_rows.append(r"\end{table}")
    
    latex_str = "\n".join(latex_rows)
    
    with open(output_path, 'w') as f:
        f.write(latex_str)
    
    print("LaTeX Table:")
    print(latex_str)
    print(f"\n[Saved] {output_path}")

export_latex_table(df_detailed, os.path.join(OUTPUT_ROOT, 'results_table.tex'))

In [ ]:
# Cell 12: Copy results to output for download
import shutil

# Copy key files to /kaggle/working for easy download
output_files = [
    "final_experiment_report.csv",
    "training_curves.png",
    "Peak_PSNR_comparison.png",
    "Final_LPIPS_comparison.png",
    "results_table.tex",
]

for f in output_files:
    src = os.path.join(OUTPUT_ROOT, f)
    if os.path.exists(src):
        shutil.copy(src, f"/kaggle/working/{f}")
        print(f"Copied: {f}")

print("\nResults available in /kaggle/working/ for download")